# Gold Metrics Validation

In [1]:
from pathlib import Path
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
catalog = SqlCatalog(
    "local",
    uri=f"sqlite:///{(DATA_DIR / 'catalog.db').resolve()}",
    warehouse=(DATA_DIR / "warehouse").resolve().as_uri(),
)

EXPECTED_VARIANTS = {
    "all",
    "exclude_zero_duration",
    "exclude_zero_distance",
    "exclude_both",
}
EXPECTED_MONTH_BASES = {"pickup_month", "source_month"}


In [2]:
GOLD_TABLES = [
    "gold.daily_trip_summary",
    "gold.monthly_kpi",
    "gold.pickup_zone_monthly",
    "gold.dropoff_zone_monthly",
    "gold.zone_flow_monthly",
    "gold.payment_monthly",
    "gold.payment_zone_flow_monthly",
    "gold.vendor_monthly",
    "gold.vendor_zone_flow_monthly",
    "gold.payment_null_profile",
    "gold.data_quality_summary",
]

def load_df(identifier):
    table = catalog.load_table(identifier)
    assert table.current_snapshot() is not None
    return table.scan().to_arrow().to_pandas()

frames = {name: load_df(name) for name in GOLD_TABLES}
pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in frames.items()
])


,table,rows,columns
0,gold.daily_trip_summary,368,13
1,gold.monthly_kpi,32,14
2,gold.pickup_zone_monthly,6276,12
3,gold.dropoff_zone_monthly,6276,12
4,gold.zone_flow_monthly,962876,13
5,gold.payment_monthly,140,14
6,gold.payment_zone_flow_monthly,1893526,14
7,gold.vendor_monthly,92,12
8,gold.vendor_zone_flow_monthly,1696972,14
9,gold.payment_null_profile,100,8


In [3]:
for name in GOLD_TABLES:
    if name != "gold.data_quality_summary":
        assert set(frames[name]["data_variant"].dropna().unique()) == EXPECTED_VARIANTS

monthly_tables = [
    "gold.monthly_kpi",
    "gold.pickup_zone_monthly",
    "gold.dropoff_zone_monthly",
    "gold.zone_flow_monthly",
    "gold.payment_monthly",
    "gold.payment_zone_flow_monthly",
    "gold.vendor_monthly",
    "gold.vendor_zone_flow_monthly",
]
for name in monthly_tables:
    assert set(frames[name]["month_basis"].dropna().unique()) == EXPECTED_MONTH_BASES

quality_bases = set(frames["gold.data_quality_summary"]["month_basis"].dropna().unique())
assert "overall" in quality_bases
assert EXPECTED_MONTH_BASES.issubset(quality_bases)
print("Dimensions are valid.")


Dimensions are valid.


In [4]:
monthly = frames["gold.monthly_kpi"]
for basis in EXPECTED_MONTH_BASES:
    pivot = monthly[monthly["month_basis"] == basis].pivot(
        index="year_month",
        columns="data_variant",
        values="trip_count",
    )
    assert (pivot["all"] >= pivot["exclude_zero_duration"]).all()
    assert (pivot["all"] >= pivot["exclude_zero_distance"]).all()
    assert (pivot["exclude_zero_duration"] >= pivot["exclude_both"]).all()
    assert (pivot["exclude_zero_distance"] >= pivot["exclude_both"]).all()
print("Variant ordering is valid.")


Variant ordering is valid.


In [5]:
daily = frames["gold.daily_trip_summary"]
pickup_monthly = monthly[monthly["month_basis"] == "pickup_month"]

daily_total = daily.groupby("data_variant")["trip_count"].sum()
monthly_total = pickup_monthly.groupby("data_variant")["trip_count"].sum()
assert daily_total.equals(monthly_total)
print("Daily and pickup-month totals reconcile.")


Daily and pickup-month totals reconcile.


In [6]:
keys = ["data_variant", "month_basis", "year_month"]
pickup = frames["gold.pickup_zone_monthly"].groupby(keys)["trip_count"].sum()
dropoff = frames["gold.dropoff_zone_monthly"].groupby(keys)["trip_count"].sum()
flow = frames["gold.zone_flow_monthly"].groupby(keys)["trip_count"].sum()
assert pickup.equals(flow)
assert dropoff.equals(flow)
print("Zone aggregates reconcile.")


Zone aggregates reconcile.


In [7]:
profile = frames["gold.payment_null_profile"]
profile[
    (profile["data_variant"] == "all")
    & (profile["payment_type"] == 0)
]


,data_variant,payment_type,field_name,row_count,null_count,null_percent,all_profile_fields_null_count,all_profile_fields_null_percent
0,all,0,airport_fee,3056852,3056852,100.0,3056852,100.0
1,all,0,congestion_surcharge,3056852,3056852,100.0,3056852,100.0
2,all,0,passenger_count,3056852,3056852,100.0,3056852,100.0
3,all,0,ratecode_id,3056852,3056852,100.0,3056852,100.0
4,all,0,store_and_fwd_flag,3056852,3056852,100.0,3056852,100.0


In [8]:
quality = frames["gold.data_quality_summary"]
quality.groupby(
    ["month_basis", "metric_category", "metric_name"],
    dropna=False,
).size().reset_index(name="rows")


,month_basis,metric_category,metric_name,rows
0,overall,month_offset,offset_-1,1
1,overall,month_offset,offset_0,1
2,overall,month_offset,offset_1,1
3,overall,month_offset_file,offset_-1,3
4,overall,month_offset_file,offset_0,3
5,overall,month_offset_file,offset_1,3
6,overall,pipeline,bronze_rows,1
7,overall,pipeline,rows_accounted_for,1
8,overall,pipeline,rows_rejected_unique,1
9,overall,pipeline,silver_rows,1
